# Week 16, Day 3: Bayesian perception: combining cues and priors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w016/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Combine two cues

Write `combine(mu1, s1, mu2, s2)` returning the combined mean and standard deviation. Check it by multiplying the two Gaussian densities on a fine grid, normalising, and computing the mean `grid_mu` and standard deviation `grid_sd`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def combine(mu1, s1, mu2, s2):
    w1 = ...
    mu = ...
    s = ...
    return mu, s

x = np.linspace(30, 80, 20001)
gauss = lambda x, m, s: np.exp(-(x - m) ** 2 / (2 * s ** 2))
prod = gauss(x, 50.0, 1.0) * gauss(x, 56.0, 2.0)
prod /= prod.sum()
grid_mu = ...
grid_sd = ...
print("formula:", np.round(combine(50.0, 1.0, 56.0, 2.0), 4), "  grid:", round(grid_mu, 4), round(grid_sd, 4))

In [ ]:
import numpy as np
def test_formula():
    "combine gives the precision-weighted mean and the combined sd"
    mu, s = combine(50.0, 1.0, 56.0, 2.0)
    assert abs(mu - (0.8 * 50 + 0.2 * 56)) < 1e-9 and abs(s - np.sqrt(1 / (1 + 0.25))) < 1e-9

def test_grid():
    "the product of the two densities on a grid has the same mean and sd"
    assert abs(grid_mu - combine(50.0, 1.0, 56.0, 2.0)[0]) < 1e-3 and abs(grid_sd - combine(50.0, 1.0, 56.0, 2.0)[1]) < 1e-3

test_formula()
test_grid()
print("All checks passed")

### Exercise: An observer in the experiment

For four levels of visual noise, simulate 5,000 conflict trials in which vision reports the true height minus 1.5 mm and touch the true height plus 1.5 mm, each with its own noise. The observer combines them optimally. Measure the visual weight from the mean estimate, store it in `w_measured`, and compare `sd_optimal` with the better single cue's sd, `sd_best_single`.

In [ ]:
rng = np.random.default_rng(2002)
sigma_H = 2.0
sigma_V = np.array([0.67, 1.33, 2.0, 4.0])
true, conflict, T = 55.0, 3.0, 5000

w_measured, w_predicted, sd_optimal, sd_best_single = [], [], [], []
for sV in sigma_V:
    xV = true - conflict / 2 + sV * rng.normal(size=T)
    xH = true + conflict / 2 + sigma_H * rng.normal(size=T)
    est, _ = combine(xV, sV, xH, sigma_H)                 # works on arrays too
    # mean estimate = wV (true - c/2) + (1 - wV)(true + c/2), so wV = (true + c/2 - mean) / c
    w_measured.append(...)
    w_predicted.append(...)
    sd_optimal.append(est.std())
    sd_best_single.append(min(sV, sigma_H))
w_measured, w_predicted, sd_optimal, sd_best_single = map(np.array, (w_measured, w_predicted, sd_optimal, sd_best_single))
print("visual sd:        ", sigma_V); print("weight measured:  ", w_measured.round(3)); print("weight predicted: ", w_predicted.round(3))
print("sd optimal:       ", sd_optimal.round(3)); print("sd better cue:    ", sd_best_single.round(3))

In [ ]:
import numpy as np
def test_weights():
    "the measured visual weights match the predicted weights to within 0.05"
    assert np.all(np.abs(w_measured - w_predicted) < 0.05)

def test_variance():
    "the optimal observer's sd is below the better single cue's at every noise level, most of all when the cues are equally reliable"
    assert np.all(sd_optimal < sd_best_single)
    i = list(sigma_V).index(2.0)
    assert sd_optimal[i] < 0.75 * sd_best_single[i]

test_weights()
test_variance()
print("All checks passed")

### Exercise: Blur flattens the slope

Shifts come from a prior with mean 1 cm and sd 0.5 cm. For feedback noise of 0.1, 0.5, 1 and 2 cm, simulate 4,000 trials, compute the Bayesian estimate of each shift, and fit the slope of the estimate against the true shift. Store the fitted and predicted slopes in `slopes` and `predicted`.

In [ ]:
mu_p, sigma_p = 1.0, 0.5
noise_levels = np.array([0.1, 0.5, 1.0, 2.0])
slopes, predicted = [], []
for sl in noise_levels:
    s = mu_p + sigma_p * rng.normal(size=4000)          # true shifts
    x = s + sl * rng.normal(size=4000)                   # blurred feedback
    est, _ = combine(x, sl, mu_p, sigma_p)               # likelihood and prior are both cues
    slopes.append(...)
    predicted.append(...)
slopes, predicted = np.array(slopes), np.array(predicted)
print("feedback noise:", noise_levels, " slopes:", slopes.round(3), " predicted:", predicted.round(3))

In [ ]:
import numpy as np
def test_slopes():
    "the fitted slopes match the predicted sigma_p^2 / (sigma_p^2 + sigma_l^2) to within 0.03"
    assert np.all(np.abs(slopes - predicted) < 0.03)

def test_order():
    "slopes fall as the feedback gets noisier"
    assert np.all(np.diff(slopes) < 0)

test_slopes()
test_order()
print("All checks passed")